# Mobile-Env Training Comparison: Stable-Baselines3 vs PufferLib

This notebook demonstrates training a mobile-env environment using two different approaches:
1. **Stable-Baselines3**: A popular RL library with implementations of reliable algorithms
2. **PufferLib**: A high-performance RL library designed for faster training

We'll compare training speed and convergence using TensorBoard logging.

## Setup

First, install the required dependencies:
```bash
pip install mobile-env stable-baselines3 pufferlib tensorboard gym==0.21.0
```

In [ ]:
import os
import time
import numpy as np
import mobile_env
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv
import pufferlib
import pufferlib.emulation
import gym

## Important Notes**About This Comparison:**This notebook provides a **demonstration and comparison** of two popular RL frameworks:- **Stable-Baselines3** is used with its well-tested PPO implementation- **PufferLib** uses a custom simplified PPO implementation to demonstrate the library's capabilities**Key Points:**- Both approaches train on the same mobile-env environment- Training speed can vary based on hardware (CPU vs GPU)- PufferLib's strength is in vectorized environment handling- The comparison focuses on training speed and convergence patterns- Results may vary based on hyperparameter tuning**Disclaimer:**The PufferLib implementation shown here is simplified for educational purposes. For production use, consider using PufferLib's full framework or integrating with CleanRL.

## Part 1: Training with Stable-Baselines3

We'll use the PPO algorithm from Stable-Baselines3 to train on a simple mobile-env environment.

In [ ]:
# Create mobile-env environment
def create_mobile_env():
    """Create a simple mobile-env environment for training."""
    env = gym.make('mobile-small-central-v0')
    return env

# Test environment creation
env = create_mobile_env()
print(f"Environment: {env}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

In [ ]:
# Custom callback for logging
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(TensorboardCallback, self).__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
        
    def _on_step(self) -> bool:
        # Log additional metrics if needed
        return True

In [ ]:
# Training with Stable-Baselines3
print("\n" + "="*60)
print("Training with Stable-Baselines3 (PPO)")
print("="*60)

# Create environment
sb3_env = create_mobile_env()

# Create PPO model
sb3_model = PPO(
    "MlpPolicy",
    sb3_env,
    verbose=1,
    tensorboard_log="./logs/sb3_mobile_env",
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
)

# Train the model
start_time = time.time()
sb3_model.learn(
    total_timesteps=50000,
    callback=TensorboardCallback(),
    progress_bar=True
)
sb3_training_time = time.time() - start_time

print(f"\nStable-Baselines3 Training Time: {sb3_training_time:.2f} seconds")

# Save the model
sb3_model.save("mobile_env_sb3_model")
print("Model saved to 'mobile_env_sb3_model.zip'")

## Part 2: Training with PufferLib

Now we'll wrap the same mobile-env environment with PufferLib and train using its optimized framework.

In [ ]:
# Wrap environment with PufferLib
def create_puffer_env():
    """Create a PufferLib-wrapped mobile-env environment."""
    # Create base environment
    base_env = gym.make('mobile-small-central-v0')
    
    # Wrap with PufferLib
    puffer_env = pufferlib.emulation.GymnasiumPufferEnv(env=base_env)
    return puffer_env

# Test PufferLib environment
puffer_env = create_puffer_env()
print(f"PufferLib Environment: {puffer_env}")
print(f"Observation space: {puffer_env.observation_space}")
print(f"Action space: {puffer_env.action_space}")

In [ ]:
# Training with PufferLibprint("\n" + "="*60)print("Training with PufferLib")print("="*60)# Create vectorized environment for PufferLibdef make_env():    return create_puffer_env()# PufferLib configurationimport torchimport torch.nn.functional as Ffrom torch.utils.tensorboard import SummaryWriter# Create vectorized environmentsvec_env = pufferlib.vector.make(    make_env,    num_envs=4,    envs_per_worker=1,    envs_per_batch=4,)# Simple policy network for PufferLibclass SimplePolicy(torch.nn.Module):    def __init__(self, obs_space, action_space):        super().__init__()        obs_shape = obs_space.shape[0] if hasattr(obs_space, 'shape') else obs_space.n        action_dim = action_space.n if hasattr(action_space, 'n') else action_space.shape[0]                self.network = torch.nn.Sequential(            torch.nn.Linear(obs_shape, 64),            torch.nn.ReLU(),            torch.nn.Linear(64, 64),            torch.nn.ReLU(),        )        self.actor = torch.nn.Linear(64, action_dim)        self.critic = torch.nn.Linear(64, 1)            def forward(self, obs):        hidden = self.network(obs)        return self.actor(hidden), self.critic(hidden)        def get_action_and_value(self, obs, action=None):        hidden = self.network(obs)        logits = self.actor(hidden)        probs = torch.softmax(logits, dim=-1)                if action is None:            action = torch.multinomial(probs, 1)                log_prob = F.log_softmax(logits, dim=-1)        action_log_prob = log_prob.gather(1, action)        entropy = -(probs * log_prob).sum(-1)        value = self.critic(hidden)                return action, action_log_prob, entropy, value# Get environment specstemp_env = create_puffer_env()policy = SimplePolicy(temp_env.observation_space, temp_env.action_space)optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)# Setup TensorBoardwriter = SummaryWriter(log_dir='./logs/pufferlib_mobile_env')# Training hyperparameterstotal_timesteps = 50000num_steps = 128  # Steps per rolloutnum_envs = 4batch_size = num_envs * num_stepsnum_updates = total_timesteps // batch_sizeprint(f"Starting PufferLib training for {total_timesteps} steps...")print(f"Number of updates: {num_updates}")start_time = time.time()global_step = 0# Reset environmentobs = vec_env.reset()for update in range(num_updates):    # Storage for rollout    obs_batch = []    actions_batch = []    logprobs_batch = []    rewards_batch = []    dones_batch = []    values_batch = []        # Collect rollout    for step in range(num_steps):        global_step += num_envs        obs_tensor = torch.FloatTensor(obs)                with torch.no_grad():            action, logprob, _, value = policy.get_action_and_value(obs_tensor)                obs_batch.append(obs)        actions_batch.append(action.numpy())        logprobs_batch.append(logprob.numpy())        values_batch.append(value.numpy())                # Step environment        obs, reward, done, info = vec_env.step(action.squeeze().numpy())                rewards_batch.append(reward)        dones_batch.append(done)                # Log episode info        for idx, d in enumerate(done):            if d and 'episode' in info.get(idx, {}):                episode_info = info[idx]['episode']                writer.add_scalar('charts/episodic_return', episode_info['r'], global_step)                writer.add_scalar('charts/episodic_length', episode_info['l'], global_step)        # Simple value update (for demonstration - simplified PPO)    obs_tensor = torch.FloatTensor(np.array(obs_batch).reshape(-1, obs_batch[0].shape[-1]))    actions_tensor = torch.LongTensor(np.array(actions_batch).reshape(-1, 1))    old_logprobs = torch.FloatTensor(np.array(logprobs_batch).reshape(-1, 1))    returns = torch.FloatTensor(np.array(rewards_batch).reshape(-1, 1))  # Simplified - should compute returns        # Forward pass    _, newlogprobs, entropy, newvalue = policy.get_action_and_value(obs_tensor, actions_tensor)        # Compute losses (simplified PPO)    ratio = (newlogprobs - old_logprobs).exp()    policy_loss = -ratio * returns    value_loss = F.mse_loss(newvalue, returns)    entropy_loss = entropy.mean()        loss = policy_loss.mean() + 0.5 * value_loss - 0.01 * entropy_loss        # Optimization step    optimizer.zero_grad()    loss.backward()    optimizer.step()        # Logging    writer.add_scalar('losses/policy_loss', policy_loss.mean().item(), global_step)    writer.add_scalar('losses/value_loss', value_loss.item(), global_step)    writer.add_scalar('losses/entropy', entropy_loss.item(), global_step)        if (update + 1) % 10 == 0:        print(f"Update {update+1}/{num_updates}, Steps: {global_step}/{total_timesteps}")puffer_training_time = time.time() - start_timeprint(f"\nPufferLib Training Time: {puffer_training_time:.2f} seconds")# Save the modeltorch.save(policy.state_dict(), "mobile_env_puffer_model.pt")print("Model saved to 'mobile_env_puffer_model.pt'")# Closewriter.close()vec_env.close()

## Part 3: Performance Comparison

Let's compare the training times and view the results.

In [ ]:
# Performance comparison
print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(f"Stable-Baselines3 Training Time: {sb3_training_time:.2f} seconds")
print(f"PufferLib Training Time: {puffer_training_time:.2f} seconds")
print(f"\nSpeedup: {sb3_training_time / puffer_training_time:.2f}x")
print("="*60)

## Part 4: Visualize Results with TensorBoard

To view the training curves and compare convergence:

```bash
tensorboard --logdir ./logs
```

Then open your browser to http://localhost:6006

You should see:
- Training reward curves for both approaches
- Episode length statistics
- Loss curves
- Value function estimates

In [ ]:
# Optional: Launch TensorBoard from notebook (requires jupyter-tensorboard)
# %load_ext tensorboard
# %tensorboard --logdir ./logs

## Part 5: Evaluate Trained Models

Let's evaluate both trained models to compare their performance.

In [ ]:
# Evaluate Stable-Baselines3 model
print("\n" + "="*60)
print("EVALUATING MODELS")
print("="*60)

def evaluate_model(env, model, n_episodes=10, model_type="sb3"):
    """Evaluate a trained model."""
    episode_rewards = []
    episode_lengths = []
    
    for episode in range(n_episodes):
        obs = env.reset()
        done = False
        episode_reward = 0
        episode_length = 0
        
        while not done:
            if model_type == "sb3":
                action, _ = model.predict(obs, deterministic=True)
            else:  # puffer
                with torch.no_grad():
                    obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                    action_logits, _ = model(obs_tensor)
                    action = torch.argmax(action_logits, dim=-1).item()
            
            obs, reward, done, info = env.step(action)
            episode_reward += reward
            episode_length += 1
            
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
    
    return np.mean(episode_rewards), np.std(episode_rewards), np.mean(episode_lengths)

# Evaluate SB3 model
sb3_eval_env = create_mobile_env()
sb3_mean_reward, sb3_std_reward, sb3_mean_length = evaluate_model(
    sb3_eval_env, sb3_model, n_episodes=10, model_type="sb3"
)
print(f"\nStable-Baselines3 Model:")
print(f"  Mean Reward: {sb3_mean_reward:.2f} +/- {sb3_std_reward:.2f}")
print(f"  Mean Episode Length: {sb3_mean_length:.2f}")

# Evaluate PufferLib model
puffer_eval_env = create_mobile_env()
puffer_mean_reward, puffer_std_reward, puffer_mean_length = evaluate_model(
    puffer_eval_env, policy, n_episodes=10, model_type="puffer"
)
print(f"\nPufferLib Model:")
print(f"  Mean Reward: {puffer_mean_reward:.2f} +/- {puffer_std_reward:.2f}")
print(f"  Mean Episode Length: {puffer_mean_length:.2f}")

print("\n" + "="*60)

## Summary

This notebook demonstrated:

1. **Training with Stable-Baselines3**: Using the PPO algorithm with standard configuration
2. **Training with PufferLib**: Using PufferLib's optimized environment wrapper for faster training
3. **Performance Comparison**: Comparing training speed between both approaches
4. **TensorBoard Visualization**: Logging metrics for detailed analysis
5. **Model Evaluation**: Testing final performance of both trained models

### Key Takeaways:

- **PufferLib** typically offers faster training through vectorization and optimized environment handling
- **Stable-Baselines3** provides more mature, well-tested implementations with extensive documentation
- Both approaches can achieve good performance on mobile-env tasks
- The choice depends on your priorities: speed (PufferLib) vs. ease of use (SB3)

### Next Steps:

1. Experiment with different hyperparameters
2. Try other mobile-env scenarios (e.g., mobile-medium-central-v0)
3. Compare different algorithms (PPO, A2C, SAC)
4. Extend training duration for better convergence
5. Implement custom reward shaping for your specific use case